In [19]:
import math

import numpy as np
import pandas as pd
import plotly.express as px
import random

In [20]:
#inital request's probability of being valid given data, descriptors or input
pval = 1

#component performance
p_c = .75

pi = np.array([[pval], [1-pval]])
C = np.array([[p_c, 0], [1-p_c, 1]])

In [21]:
#observed component performance
C @ pi

array([[0.75],
       [0.25]])

In [22]:
#building 

rng = np.random.default_rng()

n_components = 6


def build_component_matrix(p_c):
    return np.array(np.array([[p_c, 0], [1-p_c, 1]]))


component_ensemble = {}

for i in range(n_components):
    component_id = f"C{i + 1}"
    p_c = rng.uniform(low=0.5, high=0.8)
    component_ensemble[component_id] = build_component_matrix(p_c)


In [23]:
n_users = 100


def build_human_matrix(p_ac, p_ai):
    return np.array([[p_ac, p_ai], [1 - p_ac, 1 - p_ai]])


user_ensemble = {}

for i in range(n_users):
    user_id = f"u{i + 1}"
    p_ac = rng.uniform(low=0.8, high=1.0)
    p_ai = rng.uniform(low=0.0, high=0.2)
    # p_ac = 1
    # p_ai = 0
    user_ensemble[user_id] = build_human_matrix(p_ac, p_ai)

In [24]:
def link_system_components(ensemble, component_list):
    feature_dict = {}
    result = np.eye(2)
    for key in component_list:
        isSuccess = rng.random() < ensemble[key][0][0]
        feature_dict[key] = isSuccess
        result = np.matmul(result, build_component_matrix(isSuccess))
        
    return result, feature_dict

In [25]:
def simulate_daily_activity(day, n_items, user_ensemble, component_ensemble):
    pi = np.array([[1],[0]])
    telemetry = []
    user_ids = list(user_ensemble.keys())
    component_ids = list(component_ensemble.keys())

    for _ in range(n_items):
        user = rng.choice(user_ids)
        system_inference_path = random.sample(component_ids, random.randint(2,6))
        composite_system_performance, feature_dict = link_system_components(component_ensemble, system_inference_path)
        accept_prob = float((user_ensemble[user] @ composite_system_performance @ pi)[0, 0])
        is_accepted = rng.random() < accept_prob
        telemetry.append(
            [day, user, is_accepted, system_inference_path, feature_dict]
        )

    return pd.DataFrame(
        telemetry,
        columns=[
            "Date",
            "User",
            "isAccepted",
            "systemInferencePath",
            "featuresDict",
        ],
    )

In [26]:
#improvement rate measures as (1-c11)*r per execusion (cadence controlled in the simulation step)
def component_improvement_sprint(component_ensemble, component_list, improvement_rate):
    for key in component_list:
        improvement = (1-component_ensemble[key][0][0])*improvement_rate
        component_ensemble[key][0][0] += improvement
        component_ensemble[key][1][0] -= improvement
    return component_ensemble

In [27]:
simulate_daily_activity(1,1000,user_ensemble, component_ensemble)

,Date,User,isAccepted,systemInferencePath,featuresDict
0,1,u13,False,"[C2, C1, C6, C5, C3, C4]","{'C2': False, 'C1': True, 'C6': True, 'C5': Tr..."
1,1,u18,False,"[C5, C6, C2, C4]","{'C5': True, 'C6': True, 'C2': False, 'C4': Fa..."
2,1,u2,False,"[C6, C4, C2, C3]","{'C6': False, 'C4': True, 'C2': True, 'C3': True}"
3,1,u67,True,"[C1, C3]","{'C1': True, 'C3': True}"
4,1,u39,True,"[C5, C6]","{'C5': True, 'C6': True}"
...,...,...,...,...,...
995,1,u52,False,"[C4, C5, C2]","{'C4': False, 'C5': True, 'C2': True}"
996,1,u19,True,"[C5, C4, C3]","{'C5': True, 'C4': True, 'C3': True}"
997,1,u38,False,"[C1, C2, C3, C4, C5, C6]","{'C1': True, 'C2': False, 'C3': True, 'C4': Tr..."
998,1,u40,True,"[C1, C6]","{'C1': True, 'C6': True}"


In [28]:
n_items_per_day = 1000
simulation_duration = 365 + 90

historical_telemetry = []

for day_idx in range(simulation_duration):
    # system_correct_prob = min(system_quality_over_time(day_idx), 1.0)
    historical_telemetry.append(
        simulate_daily_activity(
            day=day_idx + 1,
            n_items=n_items_per_day,
            user_ensemble=user_ensemble,
            component_ensemble=component_ensemble,
        )
    )

    if (day_idx + 1) % 7 == 0:
        if day_idx < 365 + 60:
            component_ensemble = component_improvement_sprint(component_ensemble, random.sample(list(component_ensemble.keys()), 1), .4)

historical_telemetry_df = pd.concat(historical_telemetry, ignore_index=True)
historical_telemetry_df.head()

,Date,User,isAccepted,systemInferencePath,featuresDict
0,1,u40,False,"[C4, C1, C2, C6, C5, C3]","{'C4': True, 'C1': False, 'C2': True, 'C6': Tr..."
1,1,u89,False,"[C1, C6, C4, C3, C2, C5]","{'C1': False, 'C6': True, 'C4': True, 'C3': Tr..."
2,1,u52,False,"[C4, C6, C1, C3, C2, C5]","{'C4': True, 'C6': True, 'C1': False, 'C3': Tr..."
3,1,u92,False,"[C6, C5, C3, C1, C4, C2]","{'C6': False, 'C5': True, 'C3': True, 'C1': Tr..."
4,1,u32,False,"[C3, C1, C4, C2, C5]","{'C3': True, 'C1': True, 'C4': False, 'C2': Tr..."


In [29]:
historical_telemetry_df.systemInferencePath.apply(lambda x: x.sort())

0         None
1         None
2         None
3         None
4         None
          ... 
454995    None
454996    None
454997    None
454998    None
454999    None
Name: systemInferencePath, Length: 455000, dtype: object

In [30]:
observed_daily_accepts = (
    historical_telemetry_df.groupby(["Date"])["isAccepted"]
    .mean()
    .reset_index()
)

fig = px.scatter(
    observed_daily_accepts,
    x="Date",
    y="isAccepted"
)
fig.update_traces(marker={"size": 5})
fig.update_layout(yaxis_tickformat=".0%")
fig.show()

In [31]:
# expanded_df = historical_telemetry_df.systemInferencePath.apply(pd.Series)

expanded_df = pd.json_normalize(historical_telemetry_df.featuresDict)

# Optional: Rename the new columns (e.g., C_1, C_2...)
# expanded_df = expanded_df.rename(columns=lambda x: f'Component_{x+1}')

In [32]:
# expanded_df = ~expanded_df.isna()

In [33]:
list(expanded_df.columns)

['C4', 'C1', 'C2', 'C6', 'C5', 'C3']

In [34]:
historical_telemetry_df[list(expanded_df.columns)] = expanded_df

In [35]:
categories = {}
cc = 1
for inf_path in historical_telemetry_df.systemInferencePath.astype('str').unique():
    categories[inf_path] = "Type_" + str(cc)
    cc += 1

In [36]:
historical_telemetry_df["request_type"] = historical_telemetry_df.systemInferencePath.apply(lambda x: categories[str(x)])

In [37]:
historical_telemetry_df

,Date,User,isAccepted,systemInferencePath,featuresDict,C4,C1,C2,C6,C5,C3,request_type
0,1,u40,False,"[C1, C2, C3, C4, C5, C6]","{'C4': True, 'C1': False, 'C2': True, 'C6': Tr...",True,False,True,True,True,False,Type_1
1,1,u89,False,"[C1, C2, C3, C4, C5, C6]","{'C1': False, 'C6': True, 'C4': True, 'C3': Tr...",True,False,False,True,False,True,Type_1
2,1,u52,False,"[C1, C2, C3, C4, C5, C6]","{'C4': True, 'C6': True, 'C1': False, 'C3': Tr...",True,False,True,True,True,True,Type_1
3,1,u92,False,"[C1, C2, C3, C4, C5, C6]","{'C6': False, 'C5': True, 'C3': True, 'C1': Tr...",True,True,False,False,True,True,Type_1
4,1,u32,False,"[C1, C2, C3, C4, C5]","{'C3': True, 'C1': True, 'C4': False, 'C2': Tr...",False,True,True,NaN,True,True,Type_2
...,...,...,...,...,...,...,...,...,...,...,...,...
454995,455,u12,True,"[C1, C2, C5, C6]","{'C5': True, 'C2': True, 'C6': True, 'C1': True}",NaN,True,True,True,True,NaN,Type_15
454996,455,u50,False,"[C1, C3, C5]","{'C5': True, 'C3': True, 'C1': True}",NaN,True,NaN,NaN,True,True,Type_52
454997,455,u53,True,"[C3, C4, C5, C6]","{'C5': True, 'C3': True, 'C4': True, 'C6': True}",True,NaN,NaN,True,True,True,Type_38
454998,455,u73,True,"[C1, C2, C3, C5, C6]","{'C5': True, 'C1': True, 'C3': True, 'C2': Tru...",NaN,True,True,True,True,True,Type_41


In [38]:
daily_accept_stats = (
    historical_telemetry_df.groupby(["Date", "request_type"], as_index=False)
    .agg(
        accepted_sum=("isAccepted", "sum"),
        n_obs=("isAccepted", "size"),
    )
)

if pd.api.types.is_datetime64_any_dtype(daily_accept_stats["Date"]):
    all_dates = pd.date_range(
        daily_accept_stats["Date"].min(),
        daily_accept_stats["Date"].max(),
        freq="D",
        name="Date",
    )
else:
    all_dates = pd.Index(
        range(daily_accept_stats["Date"].min(), daily_accept_stats["Date"].max() + 1),
        name="Date",
    )

observed_daily_accepts = (
    daily_accept_stats.set_index(["request_type", "Date"])
    .reindex(
        pd.MultiIndex.from_product(
            [daily_accept_stats["request_type"].unique(), all_dates],
            names=["request_type", "Date"],
        ),
        fill_value=0,
    )
    .reset_index()
    .sort_values(["request_type", "Date"])
)

observed_daily_accepts[["accepted_sum_7d", "n_obs_7d"]] = (
    observed_daily_accepts.groupby("request_type")[["accepted_sum", "n_obs"]]
    .transform(lambda s: s.rolling(window=7, min_periods=1).sum())
)

observed_daily_accepts["isAccepted"] = (
    observed_daily_accepts["accepted_sum_7d"] / observed_daily_accepts["n_obs_7d"]
)

observed_daily_accepts = observed_daily_accepts[observed_daily_accepts["n_obs"] > 0]

fig = px.scatter(
    observed_daily_accepts,
    x="Date",
    y="isAccepted",
    color="request_type",
    hover_name="request_type",
    opacity=0.45,
    title="7-Day Trailing Acceptance Rate by Request Type",
    labels={"isAccepted": "Acceptance Rate", "Date": "Day"},
)
fig.update_traces(marker={"size": 5})
# fig.update_layout(yaxis_tickformat=".0%")
fig.show()

In [39]:
select_telemetry = historical_telemetry_df.loc[(historical_telemetry_df.Date <= historical_telemetry_df.Date.max()) & (historical_telemetry_df.Date >= historical_telemetry_df.Date.max() - 30)]

In [40]:
select_telemetry

,Date,User,isAccepted,systemInferencePath,featuresDict,C4,C1,C2,C6,C5,C3,request_type
424000,425,u65,True,"[C1, C2, C3, C4, C5, C6]","{'C4': True, 'C1': True, 'C2': True, 'C3': Tru...",True,True,True,True,True,True,Type_1
424001,425,u100,False,"[C1, C3, C5, C6]","{'C3': True, 'C5': True, 'C1': True, 'C6': True}",NaN,True,NaN,True,True,True,Type_7
424002,425,u12,True,"[C2, C3, C5]","{'C3': True, 'C5': True, 'C2': True}",NaN,NaN,True,NaN,True,True,Type_37
424003,425,u63,True,"[C3, C4, C5, C6]","{'C6': True, 'C5': True, 'C3': True, 'C4': True}",True,NaN,NaN,True,True,True,Type_38
424004,425,u99,True,"[C1, C2, C4, C5, C6]","{'C2': True, 'C5': True, 'C1': True, 'C4': Tru...",True,True,True,True,True,NaN,Type_39
...,...,...,...,...,...,...,...,...,...,...,...,...
454995,455,u12,True,"[C1, C2, C5, C6]","{'C5': True, 'C2': True, 'C6': True, 'C1': True}",NaN,True,True,True,True,NaN,Type_15
454996,455,u50,False,"[C1, C3, C5]","{'C5': True, 'C3': True, 'C1': True}",NaN,True,NaN,NaN,True,True,Type_52
454997,455,u53,True,"[C3, C4, C5, C6]","{'C5': True, 'C3': True, 'C4': True, 'C6': True}",True,NaN,NaN,True,True,True,Type_38
454998,455,u73,True,"[C1, C2, C3, C5, C6]","{'C5': True, 'C1': True, 'C3': True, 'C2': Tru...",NaN,True,True,True,True,True,Type_41


In [41]:
from sklearn.model_selection import train_test_split
from catboost import CatBoostClassifier
import numpy as np
import pandas as pd

df = select_telemetry.copy()
y = df["isAccepted"].astype(int)
X = df[['User']]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y
)

cat_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

model = CatBoostClassifier(
    loss_function="Logloss",
    eval_metric="AUC",
    auto_class_weights="Balanced",  # good for your 7.5% positive rate
    depth=6,
    learning_rate=0.05,
    iterations=2000,
    random_seed=42,
    verbose=200
)

model.fit(
    X_train, y_train,
    cat_features=cat_cols,          # <--- names, not indices
    eval_set=(X_test, y_test),
    use_best_model=True
)

0:	test: 0.6255855	best: 0.6255855 (0)	total: 83.6ms	remaining: 2m 47s
200:	test: 0.6331418	best: 0.6370645 (43)	total: 1.6s	remaining: 14.4s
400:	test: 0.6336516	best: 0.6370645 (43)	total: 3.19s	remaining: 12.7s
600:	test: 0.6329892	best: 0.6370645 (43)	total: 4.71s	remaining: 11s
800:	test: 0.6332132	best: 0.6370645 (43)	total: 6.24s	remaining: 9.34s
1000:	test: 0.6325663	best: 0.6370645 (43)	total: 7.71s	remaining: 7.69s
1200:	test: 0.6325663	best: 0.6370645 (43)	total: 9.12s	remaining: 6.07s
1400:	test: 0.6324915	best: 0.6370645 (43)	total: 10.5s	remaining: 4.51s
1600:	test: 0.6324915	best: 0.6370645 (43)	total: 11.9s	remaining: 2.97s
1800:	test: 0.6324698	best: 0.6370645 (43)	total: 13.3s	remaining: 1.47s
1999:	test: 0.6324698	best: 0.6370645 (43)	total: 14.6s	remaining: 0us

bestTest = 0.6370644573
bestIteration = 43

Shrink model to first 44 iterations.


In [42]:
from sklearn.model_selection import train_test_split
from catboost import CatBoostClassifier
import numpy as np
import pandas as pd

df = select_telemetry.copy()
y = df["isAccepted"].astype(int)
X = df[['Date','User']]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y
)

cat_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

model = CatBoostClassifier(
    loss_function="Logloss",
    eval_metric="AUC",
    auto_class_weights="Balanced",  # good for your 7.5% positive rate
    depth=6,
    learning_rate=0.05,
    iterations=1000,
    # random_seed=42,
    verbose=200
)

model.fit(
    X_train, y_train,
    cat_features=cat_cols,          # <--- names, not indices
    eval_set=(X_test, y_test),
    use_best_model=True
)

0:	test: 0.6206387	best: 0.6206387 (0)	total: 5.57ms	remaining: 5.57s
200:	test: 0.6286888	best: 0.6286888 (198)	total: 1.63s	remaining: 6.48s
400:	test: 0.6232486	best: 0.6287730 (211)	total: 3.51s	remaining: 5.24s
600:	test: 0.6208121	best: 0.6287730 (211)	total: 5.26s	remaining: 3.5s
800:	test: 0.6185559	best: 0.6287730 (211)	total: 7.05s	remaining: 1.75s
999:	test: 0.6159426	best: 0.6287730 (211)	total: 8.83s	remaining: 0us

bestTest = 0.628773045
bestIteration = 211

Shrink model to first 212 iterations.


In [43]:
from sklearn.model_selection import train_test_split
from catboost import CatBoostClassifier
import numpy as np
import pandas as pd

df = select_telemetry.copy()
y = df["isAccepted"].astype(int)
X = df[['Date','User', 'request_type']]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y
)

cat_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()


model = CatBoostClassifier(
    loss_function="Logloss",
    eval_metric="AUC",
    auto_class_weights="Balanced",
    depth=6,
    learning_rate=0.05,
    iterations=1000,
    # random_seed=42,
    verbose=200
)

model.fit(
    X_train, y_train,
    cat_features=cat_cols,
    eval_set=(X_test, y_test),
    use_best_model=True
)

0:	test: 0.6216652	best: 0.6216652 (0)	total: 7.28ms	remaining: 7.27s
200:	test: 0.6251354	best: 0.6333747 (29)	total: 1.79s	remaining: 7.13s
400:	test: 0.6240230	best: 0.6333747 (29)	total: 3.78s	remaining: 5.64s
600:	test: 0.6205004	best: 0.6333747 (29)	total: 5.84s	remaining: 3.88s
800:	test: 0.6179863	best: 0.6333747 (29)	total: 7.84s	remaining: 1.95s
999:	test: 0.6164066	best: 0.6333747 (29)	total: 10.1s	remaining: 0us

bestTest = 0.6333746981
bestIteration = 29

Shrink model to first 30 iterations.


In [45]:
from sklearn.model_selection import train_test_split
from catboost import CatBoostClassifier
import numpy as np
import pandas as pd

df = select_telemetry.copy()
# df[['Component_1','Component_2', 'Component_3', 'Component_4', 'Component_5','Component_6']] = df[['Component_1','Component_2', 'Component_3', 'Component_4', 'Component_5','Component_6']].astype(int)
y = df["isAccepted"].astype(int)
X = df[['C1','C2', 'C3', 'C4', 'C5','C6', 'request_type']]
X.fillna(-1, inplace=True)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y
)

cat_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

model = CatBoostClassifier(
    loss_function="Logloss",
    eval_metric="AUC",
    auto_class_weights="Balanced",
    depth=3,
    learning_rate=0.05,
    iterations=1000,
    random_seed=42,
    verbose=200
)

model.fit(
    X_train, y_train,
    cat_features=cat_cols,
    eval_set=(X_test, y_test),
    use_best_model=True
)

/tmp/ipykernel_36305/460466980.py:10: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



0:	test: 0.5786372	best: 0.5786372 (0)	total: 12.2ms	remaining: 12.2s
200:	test: 0.5933893	best: 0.5951430 (10)	total: 2.07s	remaining: 8.22s
400:	test: 0.5970770	best: 0.5974944 (333)	total: 3.97s	remaining: 5.93s
600:	test: 0.5964696	best: 0.5981250 (419)	total: 5.94s	remaining: 3.94s
800:	test: 0.5955394	best: 0.5981250 (419)	total: 7.87s	remaining: 1.95s
999:	test: 0.5986562	best: 0.6001898 (963)	total: 9.8s	remaining: 0us

bestTest = 0.6001898111
bestIteration = 963

Shrink model to first 964 iterations.


In [46]:
from sklearn.model_selection import train_test_split
from catboost import CatBoostClassifier
import numpy as np
import pandas as pd

df = select_telemetry.copy()
# df[['Component_1','Component_2', 'Component_3', 'Component_4', 'Component_5','Component_6']] = df[['Component_1','Component_2', 'Component_3', 'Component_4', 'Component_5','Component_6']].astype(int)
y = df["isAccepted"].astype(int)
X = df[['Date', 'User', 'C1','C2', 'C3', 'C4', 'C5','C6', 'request_type']]
X.fillna(-1, inplace=True)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y
)

cat_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

model = CatBoostClassifier(
    loss_function="Logloss",
    eval_metric="AUC",
    auto_class_weights="Balanced",
    depth=3,
    learning_rate=0.05,
    iterations=1000,
    random_seed=42,
    verbose=200
)

model.fit(
    X_train, y_train,
    cat_features=cat_cols,
    eval_set=(X_test, y_test),
    use_best_model=True
)

/tmp/ipykernel_36305/458770668.py:10: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



0:	test: 0.6833429	best: 0.6833429 (0)	total: 19.1ms	remaining: 38.2s
200:	test: 0.7060077	best: 0.7118660 (46)	total: 2.93s	remaining: 26.2s
400:	test: 0.7034092	best: 0.7118660 (46)	total: 6.2s	remaining: 24.7s
600:	test: 0.6984181	best: 0.7118660 (46)	total: 9.5s	remaining: 22.1s
800:	test: 0.6976048	best: 0.7118660 (46)	total: 12.9s	remaining: 19.3s
1000:	test: 0.6966539	best: 0.7118660 (46)	total: 16.3s	remaining: 16.2s
1200:	test: 0.6947503	best: 0.7118660 (46)	total: 19.6s	remaining: 13s
1400:	test: 0.6925433	best: 0.7118660 (46)	total: 22.9s	remaining: 9.79s
1600:	test: 0.6903267	best: 0.7118660 (46)	total: 26.3s	remaining: 6.56s
1800:	test: 0.6891089	best: 0.7118660 (46)	total: 29.7s	remaining: 3.28s
1999:	test: 0.6890814	best: 0.7118660 (46)	total: 33.1s	remaining: 0us

bestTest = 0.711866024
bestIteration = 46

Shrink model to first 47 iterations.


In [ ]:
#Seems like model path is not enough for errors to be separable even with perfect humans